# Baseline evaluation — FinVerify

This notebook runs a small, open-weights HuggingFace model (**Qwen2.5-3B-Instruct**) as the *baseline* financial advisor. It samples **5 questions from each of the three dataset subsets** (`standard_questions`, `open_ended_hard`, `reddit_questions`) and grades the answers with three signals:

1. **Multiple-choice accuracy** — exact letter match (only applies to MC items).
2. **Embedding similarity** — cosine similarity between the model answer and the reference `correct_answer`, using `BAAI/bge-small-en-v1.5`.
3. **LLM-as-judge rubric** — Claude scores the answer on `factuality`, `completeness`, and `advice_quality` (1–5 each).

Why a small open-weights baseline? The RAG notebook will use a frontier API model (Claude). Comparing Qwen-3B (no retrieval) against Claude + RAG would confound *retrieval gains* with *model-size gains*. The cleanest comparison is to also run **Claude with no retrieval** as a second baseline — we add that optionally at the bottom.

## 1. Setup

Install dependencies (uncomment the `%pip install` line on first run). Requires Python 3.10+.

The `ANTHROPIC_API_KEY` environment variable must be set for the LLM-judge step. Put it in your shell or a `.env` file; the notebook will skip the judge if it's missing and just show embedding-similarity scores.

In [1]:
import os, sys, json, time, platform, subprocess
from pathlib import Path

REPO_URL = "https://github.com/niksharma99/COMS6156FinalProject.git"
REPO_NAME = "COMS6156FinalProject"
# TODO: switch to "main" once the eval/demo branch is merged.
REPO_BRANCH = "add-evaluation-dataset"

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    clone_target = Path("/content") / REPO_NAME
    if not clone_target.exists():
        print(f"Colab detected — cloning {REPO_URL}")
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(clone_target)], check=True)
    os.chdir(clone_target)
    REPO_ROOT = clone_target
else:
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "dataset").exists():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / "dataset").exists():
        raise RuntimeError(
            f"Could not find repo root (no 'dataset/' dir found walking up from {Path.cwd()})."
        )

SRC_DIR = str(REPO_ROOT / "src")
print(f"Repo root found at: {REPO_ROOT}")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env")
except ImportError:
    print("python-dotenv not installed; relying on shell environment for API keys.")

from eval.sampling import sample_from_dataset
from eval.metrics import grade_mc, cosine_similarity, judge_with_claude, extract_mc_letter

print("Repo root:", REPO_ROOT)
print("Python:", platform.python_version())
print("In Colab:", IN_COLAB)
print("ANTHROPIC_API_KEY loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))


Repo root found at: /content/COMS6156FinalProject
Repo root: /content/COMS6156FinalProject
Python: 3.12.13
In Colab: True
ANTHROPIC_API_KEY loaded: True


## 2. Build the evaluation set

Set `MODE = "all"` to run every question in every topic file (156 items). Set `MODE = "sample"` to take a stratified N-per-dataset subset for a fast smoke test.


In [2]:
# Build the evaluation set.
#   MODE = "all"     → every question in every topic file across all three datasets
#   MODE = "sample"  → N_PER_DATASET stratified per dataset (fast smoke test)
MODE = "all"
N_PER_DATASET = 5

DATASETS = ["standard_questions", "open_ended_hard", "reddit_questions"]
TOPICS = ["budgeting", "credit_and_debt", "insurance", "investing", "retirement", "tax"]


def load_all_from_dataset(name: str) -> list[dict]:
    base = REPO_ROOT / "dataset" / name
    out = []
    for t in TOPICS:
        items = json.loads((base / f"{t}.json").read_text())
        for it in items:
            out.append({**it, "_dataset": name, "_topic": t})
    return out


items = []
if MODE == "all":
    for ds in DATASETS:
        items.extend(load_all_from_dataset(ds))
else:
    for ds in DATASETS:
        items.extend(sample_from_dataset(ds, N_PER_DATASET, seed=7))

# Per-dataset / per-type counts so you can see what you're about to run
from collections import Counter
by_dataset = Counter(i["_dataset"] for i in items)
by_type = Counter("MC" if i.get("type") == "multiple_choice" else "OE" for i in items)
print(f"MODE={MODE}  total={len(items)}")
print("  by dataset:", dict(by_dataset))
print("  by type:   ", dict(by_type))


MODE=all  total=156
  by dataset: {'standard_questions': 42, 'open_ended_hard': 42, 'reddit_questions': 72}
  by type:    {'OE': 126, 'MC': 30}


## 3. Load the baseline model

Qwen2.5-3B-Instruct runs locally on CPU / Apple Silicon (MPS) / CUDA. First load will download ~6 GB of weights to the HF cache.

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
# bfloat16 on MPS avoids fp16-triggered CPU fallbacks that make generation crawl.
# On CUDA, fp16 is fine. On CPU, fp32 is the only option.
if device == "cuda":
    dtype = torch.float16
elif device == "mps":
    dtype = torch.bfloat16
else:
    dtype = torch.float32

# Let unsupported MPS ops fall back to CPU instead of failing (surfaces slow ops instead of hanging).
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

print("Device:", device, "| dtype:", dtype)

t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL_ID)
print(f"Tokenizer loaded in {time.time()-t0:.1f}s")

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype).to(device)
model.eval()
print(f"Model loaded + moved to {device} in {time.time()-t0:.1f}s")

# Warmup: first MPS/CUDA call compiles kernels and is much slower than steady-state.
print("Warming up with a 5-token dummy generation...", flush=True)
t0 = time.time()
_warm = tok("Hello", return_tensors="pt").to(device)
with torch.no_grad():
    model.generate(**_warm, max_new_tokens=5, do_sample=False, pad_token_id=tok.eos_token_id)
print(f"Warmup done in {time.time()-t0:.1f}s")

Device: cuda | dtype: torch.float16


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Tokenizer loaded in 11.2s


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded + moved to cuda in 3.9s
Warming up with a 5-token dummy generation...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Warmup done in 0.9s


## 4. Prompt templates

For MC items we ask for a single letter so grading is mechanical. For open-ended items we ask for a concise, structured answer.

In [4]:
from transformers import TextStreamer

SYSTEM_PROMPT = (
    "You are a careful personal-finance assistant. "
    "Give direct, accurate, and nuanced answers. "
    "When tradeoffs exist, acknowledge them instead of giving one-sided advice."
)

# Adaptive token budgets: MC items need ~1 letter; OE ~300 is plenty for 4-8 sentences.
MAX_NEW_TOKENS_MC = 10
MAX_NEW_TOKENS_OE = 300


def build_prompt(item: dict) -> list[dict]:
    if item.get("type") == "multiple_choice":
        options = "\n".join(item["options"])
        user = (
            f"{item['question']}\n\n{options}\n\n"
            "Respond with ONLY the single letter (A, B, C, or D) of the best answer."
        )
    else:
        user = (
            f"{item['question']}\n\n"
            "Answer in 4-8 sentences. Be specific and mention tradeoffs where relevant."
        )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]


def generate(item: dict, max_new_tokens: int | None = None, stream: bool = True, verbose: bool = True) -> str:
    """Generate an answer with optional live token streaming and timing logs."""
    if max_new_tokens is None:
        max_new_tokens = MAX_NEW_TOKENS_MC if item.get("type") == "multiple_choice" else MAX_NEW_TOKENS_OE

    messages = build_prompt(item)

    t_tok = time.time()
    enc = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc.get("attention_mask")
    if attention_mask is not None:
        attention_mask = attention_mask.to(device)
    prompt_len = input_ids.shape[-1]
    if verbose:
        print(f"    [tokenize] {time.time()-t_tok:.2f}s  prompt_len={prompt_len}  max_new={max_new_tokens}", flush=True)

    streamer = TextStreamer(tok, skip_prompt=True, skip_special_tokens=True) if stream else None

    t_gen = time.time()
    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
            streamer=streamer,
        )
    gen_tokens = out.shape[-1] - prompt_len
    dt = time.time() - t_gen
    if verbose:
        tps = gen_tokens / dt if dt > 0 else 0.0
        print(f"\n    [generate] {dt:.2f}s  new_tokens={gen_tokens}  ({tps:.1f} tok/s)", flush=True)

    gen = out[0, prompt_len:]
    return tok.decode(gen, skip_special_tokens=True).strip()

## 5. Run the baseline (resume-safe)

Results are checkpointed to disk after each item, so you can restart the kernel without losing progress — items with an id already in the checkpoint are skipped.


In [5]:
# Run the baseline on all items in `items`. Resume-safe: if RESULTS_PATH already
# exists, previously-completed ids are skipped.
RESULTS_DIR = REPO_ROOT / "src" / "notebooks" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_SLUG = MODEL_ID.split("/")[-1]
RESULTS_PATH = RESULTS_DIR / f"baseline_{MODEL_SLUG}_all.json"

# Load prior results if present (lets you restart the kernel without losing work).
if RESULTS_PATH.exists():
    results = json.loads(RESULTS_PATH.read_text())
    print(f"Resuming: loaded {len(results)} existing results from {RESULTS_PATH.name}")
else:
    results = []
done_ids = {r["id"] for r in results}

remaining = [it for it in items if it["id"] not in done_ids]
total = len(remaining)
print(f"Running baseline on {total} remaining item(s) of {len(items)} total\n", flush=True)

t_run_start = time.time()
for i, item in enumerate(remaining, 1):
    print(f"[{i}/{total}] {item['_dataset']}/{item['_topic']}/{item['id']}  ({item.get('type', 'open_ended')})", flush=True)
    print(f"  Q: {item['question'][:120]}{'…' if len(item['question']) > 120 else ''}", flush=True)
    t0 = time.time()
    answer = generate(item, stream=False, verbose=True)
    dt = time.time() - t0
    results.append({
        "id": item["id"],
        "dataset": item["_dataset"],
        "topic": item["_topic"],
        "type": item.get("type"),
        "difficulty": item.get("difficulty"),
        "question": item["question"],
        "reference": item["correct_answer"],
        "candidate": answer,
        "latency_sec": round(dt, 2),
    })
    # Checkpoint every item so a crash costs at most one question.
    RESULTS_PATH.write_text(json.dumps(results, indent=2))
    print(f"  → {dt:.1f}s  answer[:100]: {answer[:100]}\n", flush=True)

dt_total = time.time() - t_run_start
print(f"Done. {len(results)}/{len(items)} items. This run: {dt_total:.1f}s "
      f"({dt_total/max(total,1):.1f}s/item).")
print(f"Saved to {RESULTS_PATH}")


Running baseline on 156 remaining item(s) of 156 total

[1/156] standard_questions/budgeting/bud-001  (open_ended)
  Q: What is the 50/30/20 budgeting rule?
    [tokenize] 0.01s  prompt_len=78  max_new=300

    [generate] 5.80s  new_tokens=135  (23.3 tok/s)
  → 5.8s  answer[:100]: The 50/30/20 budgeting rule is a guideline that helps individuals allocate their income into three c

[2/156] standard_questions/budgeting/bud-002  (multiple_choice)
  Q: How many months of expenses should an emergency fund cover?
    [tokenize] 0.00s  prompt_len=118  max_new=10

    [generate] 0.09s  new_tokens=2  (22.5 tok/s)
  → 0.1s  answer[:100]: B

[3/156] standard_questions/budgeting/bud-003  (open_ended)
  Q: What is the difference between a need and a want in the context of budgeting?
    [tokenize] 0.00s  prompt_len=79  max_new=300

    [generate] 6.41s  new_tokens=148  (23.1 tok/s)
  → 6.4s  answer[:100]: In the context of budgeting, distinguishing between needs and wants is crucial for effective f

## 6. Score

### 6a. Multiple-choice accuracy

In [6]:
mc_rows = [r for r in results if r["type"] == "multiple_choice"]
for r in mc_rows:
    g = grade_mc(r["candidate"], r["reference"])
    r["mc_correct"] = g["is_correct"]
    r["mc_picked"] = g["picked"]

if mc_rows:
    correct = sum(1 for r in mc_rows if r["mc_correct"])
    print(f"MC accuracy: {correct}/{len(mc_rows)} = {correct/len(mc_rows):.1%}")
else:
    print("No MC items in this sample.")

MC accuracy: 23/30 = 76.7%


### 6b. Embedding similarity (open-ended)

In [7]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
oe_rows = [r for r in results if r["type"] != "multiple_choice"]

for r in oe_rows:
    emb_ref, emb_cand = embedder.encode([r["reference"], r["candidate"]], normalize_embeddings=True)
    r["embed_cosine"] = cosine_similarity(emb_ref, emb_cand)

if oe_rows:
    import statistics
    print(f"Mean cosine vs reference: {statistics.mean(r['embed_cosine'] for r in oe_rows):.3f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Mean cosine vs reference: 0.883


### 6c. LLM-as-judge (open-ended)

Requires `ANTHROPIC_API_KEY`. If missing, this cell will be skipped.

In [8]:
if os.environ.get("ANTHROPIC_API_KEY"):
    import anthropic
    judge_client = anthropic.Anthropic()
    for r in oe_rows:
        score = judge_with_claude(
            question=r["question"],
            reference=r["reference"],
            candidate=r["candidate"],
            client=judge_client,
        )
        r["judge"] = score.to_dict()
        print(f"{r['id']:<12}  f={score.factuality} c={score.completeness} a={score.advice_quality}  mean={score.mean:.2f}")
else:
    print("ANTHROPIC_API_KEY not set — skipping LLM judge. Set it and re-run this cell.")

bud-001       f=5 c=4 a=5  mean=4.67
bud-003       f=4 c=4 a=4  mean=4.00
bud-004       f=2 c=2 a=3  mean=2.33
bud-005       f=5 c=5 a=5  mean=5.00
crd-006       f=4 c=4 a=4  mean=4.00
ins-001       f=4 c=4 a=4  mean=4.00
ins-003       f=5 c=4 a=5  mean=4.67
inv-001       f=5 c=5 a=5  mean=5.00
inv-003       f=3 c=3 a=3  mean=3.00
inv-007       f=5 c=4 a=4  mean=4.33
ret-005       f=4 c=5 a=4  mean=4.33
tax-008       f=2 c=2 a=2  mean=2.00
bud-007       f=4 c=4 a=4  mean=4.00
bud-008       f=2 c=2 a=3  mean=2.33
bud-009       f=2 c=2 a=2  mean=2.00
bud-010       f=5 c=5 a=5  mean=5.00
bud-011       f=5 c=4 a=5  mean=4.67
bud-012       f=3 c=3 a=4  mean=3.33
bud-013       f=5 c=5 a=5  mean=5.00
cd-001        f=4 c=4 a=5  mean=4.33
cd-002        f=4 c=4 a=4  mean=4.00
cd-003        f=5 c=5 a=4  mean=4.67
cd-004        f=4 c=4 a=4  mean=4.00
cd-005        f=4 c=5 a=5  mean=4.67
cd-006        f=4 c=4 a=4  mean=4.00
cd-007        f=4 c=3 a=4  mean=3.67
ins-001       f=4 c=3 a=4  mean=3.67
i

## 7. Summary tables

In [9]:
import pandas as pd
df = pd.DataFrame(results)
df.head()

,id,dataset,topic,type,difficulty,question,reference,candidate,latency_sec,embed_cosine,judge,mc_correct,mc_picked
0,bud-001,standard_questions,budgeting,open_ended,basic,What is the 50/30/20 budgeting rule?,The 50/30/20 rule suggests allocating your aft...,The 50/30/20 budgeting rule is a guideline tha...,5.81,0.862503,"{'factuality': 5, 'completeness': 4, 'advice_q...",NaN,NaN
1,bud-002,standard_questions,budgeting,multiple_choice,basic,How many months of expenses should an emergenc...,B,B,0.09,NaN,NaN,True,B
2,bud-003,standard_questions,budgeting,open_ended,basic,What is the difference between a need and a wa...,A need is an essential expense required for ba...,"In the context of budgeting, distinguishing be...",6.42,0.889655,"{'factuality': 4, 'completeness': 4, 'advice_q...",NaN,NaN
3,bud-004,standard_questions,budgeting,open_ended,intermediate,Should you prioritize paying off high-interest...,Most financial advisors recommend a balanced a...,When deciding between paying off high-interest...,6.93,0.873562,"{'factuality': 2, 'completeness': 2, 'advice_q...",NaN,NaN
4,bud-005,standard_questions,budgeting,open_ended,intermediate,What is the debt avalanche method vs. the debt...,The debt avalanche method prioritizes paying o...,The debt avalanche method and the debt snowbal...,9.23,0.963314,"{'factuality': 5, 'completeness': 5, 'advice_q...",NaN,NaN


In [10]:
# Per-dataset aggregates
def agg(g):
    row = {"n": len(g)}
    if "mc_correct" in g.columns:
        mc = g.dropna(subset=["mc_correct"])
        row["mc_accuracy"] = mc["mc_correct"].mean() if len(mc) else None
    if "embed_cosine" in g.columns:
        row["mean_cosine"] = g["embed_cosine"].mean()
    if "judge" in g.columns:
        judged = g.dropna(subset=["judge"])
        if len(judged):
            row["judge_factuality"] = judged["judge"].apply(lambda j: j["factuality"]).mean()
            row["judge_completeness"] = judged["judge"].apply(lambda j: j["completeness"]).mean()
            row["judge_advice"] = judged["judge"].apply(lambda j: j["advice_quality"]).mean()
            row["judge_mean"] = judged["judge"].apply(lambda j: j["mean"]).mean()
    row["mean_latency_sec"] = g["latency_sec"].mean()
    return pd.Series(row)

df.groupby("dataset").apply(agg)

/tmp/ipykernel_16344/3208443437.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("dataset").apply(agg)


,n,mc_accuracy,mean_cosine,judge_factuality,judge_completeness,judge_advice,judge_mean,mean_latency_sec
dataset,,,,,,,,
open_ended_hard,42.0,NaN,0.893431,3.738095,3.809524,3.880952,3.809524,6.765476
reddit_questions,72.0,NaN,0.870248,2.458333,2.222222,2.388889,2.356481,7.292917
standard_questions,42.0,0.766667,0.922362,4.000000,3.833333,4.000000,3.944444,1.985238


## 8. Persist results (sharded by dataset / topic / type)

Writes the full `results` list plus per-dataset, per-(dataset, topic), and per-type JSON files under `src/notebooks/results/baseline_<model>/`.


In [11]:
# Shard results by dataset and by (dataset, topic) so they're easy to inspect
# and to diff against the RAG results later. Also write a CSV summary.
from collections import defaultdict

OUT_DIR = REPO_ROOT / "src" / "notebooks" / "results" / f"baseline_{MODEL_SLUG}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Combined file (same content as checkpoint above)
(OUT_DIR / "all.json").write_text(json.dumps(results, indent=2))

# Per-dataset
by_dataset: dict[str, list[dict]] = defaultdict(list)
for r in results:
    by_dataset[r["dataset"]].append(r)
for ds, rows in by_dataset.items():
    (OUT_DIR / f"{ds}.json").write_text(json.dumps(rows, indent=2))

# Per-dataset/topic
by_dt: dict[tuple[str, str], list[dict]] = defaultdict(list)
for r in results:
    by_dt[(r["dataset"], r["topic"])].append(r)
topic_dir = OUT_DIR / "by_topic"
topic_dir.mkdir(exist_ok=True)
for (ds, tp), rows in by_dt.items():
    (topic_dir / f"{ds}__{tp}.json").write_text(json.dumps(rows, indent=2))

# Per-type (MC vs OE) — handy for computing accuracy on MC only
by_type: dict[str, list[dict]] = defaultdict(list)
for r in results:
    key = "multiple_choice" if r.get("type") == "multiple_choice" else "open_ended"
    by_type[key].append(r)
for k, rows in by_type.items():
    (OUT_DIR / f"type_{k}.json").write_text(json.dumps(rows, indent=2))

print(f"Wrote {len(results)} results to {OUT_DIR}")
print(f"  datasets: {sorted(by_dataset)}")
print(f"  topic shards: {len(by_dt)}")
print(f"  types: {sorted(by_type)}")


Wrote 156 results to /content/COMS6156FinalProject/src/notebooks/results/baseline_Qwen2.5-3B-Instruct
  datasets: ['open_ended_hard', 'reddit_questions', 'standard_questions']
  topic shards: 18
  types: ['multiple_choice', 'open_ended']


## 9. (Optional) Claude-without-RAG second baseline

This isolates the contribution of retrieval. Uncomment to run once you're happy with the Qwen results.

In [ ]:
# if os.environ.get("ANTHROPIC_API_KEY"):
#     import anthropic
#     client = anthropic.Anthropic()
#     CLAUDE_MODEL = "claude-sonnet-4-6"
#     claude_results = []
#     for item in sampled:
#         msgs = build_prompt(item)
#         resp = client.messages.create(
#             model=CLAUDE_MODEL, max_tokens=600,
#             system=msgs[0]["content"],
#             messages=[{"role": "user", "content": msgs[1]["content"]}],
#         )
#         claude_results.append({
#             "id": item["id"], "dataset": item["_dataset"], "topic": item["_topic"],
#             "type": item.get("type"), "question": item["question"],
#             "reference": item["correct_answer"], "candidate": resp.content[0].text.strip(),
#         })
#     (OUT_DIR / f"baseline_{CLAUDE_MODEL}_no_rag.json").write_text(json.dumps(claude_results, indent=2))
#     print("Saved Claude no-RAG baseline.")